In [1]:
import os
import polars as pl
import gc
import shutil
from tqdm.auto import tqdm

SASREC_CAND_PATH = '/kaggle/input/datasets/b22dckh072/file04/sasrec_candidates.parquet'
LIGHTGCN_CAND_PATH = '/kaggle/input/datasets/b22dckh072/file04/lightgcn_candidates.parquet'
TEMP_DIR = '/kaggle/working/candidates_chunks_temp'
FINAL_CAND_PATH = '/kaggle/working/candidates_phase2.parquet'

os.makedirs(TEMP_DIR, exist_ok=True)

if os.path.exists(SASREC_CAND_PATH) and os.path.exists(LIGHTGCN_CAND_PATH):
    print("Đang quét cấu trúc file (Lazy Scan)...")
    
    lazy_sasrec = pl.scan_parquet(SASREC_CAND_PATH)
    lazy_lightgcn = pl.scan_parquet(LIGHTGCN_CAND_PATH)
    lazy_train = pl.scan_parquet('/kaggle/input/datasets/b22dckh072/file04/train_interactions.parquet')

    print("Đang tính toán danh sách Top 200 món đồ phổ biến nhất từ tập Train...")
    # Lấy Top 200 món đồ phổ biến để làm nguồn dự phòng
    df_popular = (
        lazy_train
        .group_by('mapped_item_id')
        .agg(pl.len().alias('pop_count'))
        .sort('pop_count', descending=True)
        .head(200)
        .select('mapped_item_id')
        .with_row_index('pop_rank') # Đánh số thứ tự phổ biến (0, 1, 2...)
        .collect()
    )

    print("Đang tính toán số lượng người dùng...")
    max_u_sasrec = lazy_sasrec.select(pl.col('mapped_user_id').max()).collect().item()
    max_u_lightgcn = lazy_lightgcn.select(pl.col('mapped_user_id').max()).collect().item()
    
    max_u_sasrec = 0 if max_u_sasrec is None else max_u_sasrec
    max_u_lightgcn = 0 if max_u_lightgcn is None else max_u_lightgcn
    max_user = max(max_u_sasrec, max_u_lightgcn)

    chunk_size = 50000 
    
    print(f"Tổng số User ID: {max_user:,}. Bắt đầu gộp và chia nhỏ file...")

    for start_u in tqdm(range(0, max_user + 1, chunk_size), desc="Đang xử lý từng phần"):
        end_u = start_u + chunk_size

        chunk_sasrec = lazy_sasrec.filter(
            (pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)
        ).collect()

        chunk_lightgcn = lazy_lightgcn.filter(
            (pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)
        ).collect()

        if chunk_sasrec.height == 0 and chunk_lightgcn.height == 0:
            continue

        # 1. Gộp 2 mô hình (RRF)
        chunk_union = chunk_sasrec.join(
            chunk_lightgcn, 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='full', 
            coalesce=True
        )
        
        chunk_train = lazy_train.filter((pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)).collect()
        
        # 2. Loại bỏ các món User đã tương tác trong Train ra khỏi RRF
        chunk_union = chunk_union.join(
            chunk_train.select(['mapped_user_id', 'mapped_item_id']), 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='anti'
        )
        
        # 3. Tính điểm RRF và lấy CỨNG Top 250
        chunk_rrf = (
            chunk_union
            .with_columns([
                (
                    pl.when(pl.col('sasrec_rank').is_not_null())
                    .then(1.0 / (60.0 + pl.col('sasrec_rank').cast(pl.Float64)))
                    .otherwise(0.0)
                    +
                    pl.when(pl.col('lightgcn_rank').is_not_null())
                    .then(1.0 / (60.0 + pl.col('lightgcn_rank').cast(pl.Float64)))
                    .otherwise(0.0)
                ).alias('rrf_score')
            ])
            .sort(['mapped_user_id', 'rrf_score'], descending=[False, True])
            .group_by('mapped_user_id', maintain_order=True)
            .head(250)
        )

        # 4. Trộn 200 đồ phổ biến cho tất cả User trong chunk hiện tại
        unique_users = chunk_rrf.select('mapped_user_id').unique()
        chunk_pop = unique_users.join(df_popular, how='cross')
        
        # Gắn điểm "âm" để đồ phổ biến luôn nằm dưới hạng 250 (-1, -2, -3...)
        # Đồng thời đồng bộ kiểu dữ liệu (schema) với bảng rrf
        chunk_pop = chunk_pop.with_columns([
            pl.lit(None).cast(chunk_rrf.schema['sasrec_rank']).alias('sasrec_rank'),
            pl.lit(None).cast(chunk_rrf.schema['lightgcn_rank']).alias('lightgcn_rank'),
            (-1.0 - pl.col('pop_rank')).cast(pl.Float64).alias('rrf_score') 
        ]).drop('pop_rank').select(chunk_rrf.columns)
        
        # 5. Nối RRF (250 items) và Phổ biến (200 items)
        chunk_combined = pl.concat([chunk_rrf, chunk_pop])
        
        # 6. Loại bỏ đồ phổ biến nếu User đó đã mua (Anti join với Train lần nữa)
        chunk_combined = chunk_combined.join(
            chunk_train.select(['mapped_user_id', 'mapped_item_id']), 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='anti'
        )
        
        # 7. Sắp xếp điểm, LỌC TRÙNG và chốt sổ Top 300
        # keep='first' đảm bảo nếu item vừa nằm ở RRF vừa Phổ biến, nó sẽ giữ nguyên điểm số cao của RRF
        chunk_final = (
            chunk_combined
            .sort(['mapped_user_id', 'rrf_score'], descending=[False, True])
            .unique(subset=['mapped_user_id', 'mapped_item_id'], keep='first', maintain_order=True)
            .group_by('mapped_user_id', maintain_order=True)
            .head(300)
            .drop('rrf_score') 
        )

        # Lưu chunk
        chunk_file_path = f"{TEMP_DIR}/cand_{start_u}_to_{end_u}.parquet"
        chunk_final.write_parquet(chunk_file_path)

        # Giải phóng bộ nhớ
        del chunk_sasrec, chunk_lightgcn, chunk_union, chunk_train, chunk_rrf, chunk_pop, chunk_combined, chunk_final, unique_users
        gc.collect()

    print("Đang hợp nhất các khối thành 1 file duy nhất...")
    pl.scan_parquet(f"{TEMP_DIR}/*.parquet").sink_parquet(FINAL_CAND_PATH)
    shutil.rmtree(TEMP_DIR)
    
    print(f"File hoàn chỉnh tại: {FINAL_CAND_PATH}")

else:
    print("Lỗi: Không tìm thấy file đầu vào. Hãy kiểm tra lại đường dẫn.")

Đang quét cấu trúc file (Lazy Scan)...
Đang tính toán danh sách Top 200 món đồ phổ biến nhất từ tập Train...
Đang tính toán số lượng người dùng...
Tổng số User ID: 2,257,153. Bắt đầu gộp và chia nhỏ file...


Đang xử lý từng phần:   0%|          | 0/46 [00:00<?, ?it/s]

Đang hợp nhất các khối thành 1 file duy nhất...
File hoàn chỉnh tại: /kaggle/working/candidates_phase2.parquet
